# Validate profiles against their resumes on a Colab GPU (Qwen 2.5 7B) - read-only

Runs `resume_collab.validate_resumes` with the same **qwen2.5:7b-instruct** (text) and **qwen2.5vl:7b** (scanned
resumes) served by Ollama as the `colab` notebook. Profiles with `specialty IS NOT NULL AND resume_url IS NOT NULL`
are compared with their resume (first name, last name, specialty, address). **Nothing is written to the database**;
findings go to three CSV files on Google Drive.

**Before you start**
1. *Runtime > Change runtime type > GPU* (T4 works; L4/A100 is faster).
2. *Secrets* (key icon), each with *Notebook access* on: `NEON_DB_URL`, `CLOUDFLARE_PUBLIC_BASE_URL`, `GITHUB_TOKEN` (read access to the repo).
3. Steps 1-4 run on every new runtime. After a disconnect, run 1-4 again, then go to **step 8** (same command as the full run).

## 1. GPU check and Google Drive (the CSV files live here)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/resume_validation'   # CSV files go to <DRIVE_DIR>/reports

## 2. Get the code (clone, or pull the latest)

In [ ]:
import os, subprocess
from google.colab import userdata
REPO = '/content/Neon-Cleaning'
BRANCH = 'main'
URL = 'https://github.com/itsupport2306/Neon-Cleaning.git'
token = userdata.get('GITHUB_TOKEN')
authed = URL.replace('https://', f'https://x-access-token:{token}@')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, authed, REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', URL], check=True)   # keep the token out of .git/config
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q', authed, BRANCH], check=True)
!git -C {REPO} log --oneline -1
%cd {REPO}

## 3. Install system packages, Ollama and Python dependencies (~3 min)

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip -q install -r requirements.txt
!ollama --version && tesseract --version | head -1

## 4. Configure, start Ollama on the GPU, load both Qwen models

In [ ]:
import sys; sys.path.insert(0, f'{REPO}/resume_collab')
import colab_runtime as rc

rc.configure(DRIVE_DIR)   # secrets + the same Ollama/Qwen settings as colab/; CSV folder = <DRIVE_DIR>/reports
rc.start_ollama()
rc.pull_models()          # ~10 GB the first time on each runtime
rc.check_gpu()            # fails if either model is not fully in GPU memory

## 5. Offline unit tests (no database or model needed)

In [ ]:
!python -m pytest -q tests/test_resume_collab.py

## 6. Preflight and eligible count (read-only)

Checks the database, proves the Qwen text model returns a valid extraction, checks OCR, and counts the profiles where `specialty IS NOT NULL AND resume_url IS NOT NULL`.

In [ ]:
rc.validate('preflight')
rc.validate('count')

## 7. Test batch: 25 profiles, then inspect the CSV files

`--limit 25` validates the next 25 profiles that are not yet recorded. Open a few resumes and check each reported discrepancy before the full run.

In [ ]:
rc.validate('run', '--limit', '25')

import pandas as pd
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 250)
OUT = rc.output_dir()
for name in ('resume_validation_discrepancies', 'resume_validation_no_discrepancies', 'resume_validation_errors'):
    df = pd.read_csv(OUT / f'{name}.csv', dtype=str)
    print(f'
=== {name}.csv: {len(df)} rows')
    display(df.head(10))

**Do not continue until the test batch looks right.** To start over, delete the three CSV files in `<DRIVE_DIR>/reports`.

## 8. Full run (also the way to resume after a stop or disconnect)

Run steps 1-4 first on a new runtime. Already-recorded profiles are skipped, so this exact cell continues where it stopped. The stop button finishes the profiles in flight, records them and exits.

In [ ]:
rc.validate('run', '--workers', '4', '--model-concurrency', '2')

## 9. Review the results

Optionally retry the retryable errors once (the old errors file is archived as `resume_validation_errors.<timestamp>.bak.csv`).

In [ ]:
rc.validate('status')
# rc.validate('run', '--retry-errors')   # uncomment to re-validate profiles that ended in the errors file

errors = pd.read_csv(OUT / 'resume_validation_errors.csv', dtype=str)
display(errors.groupby('error_category').size().rename('profiles'))
disc = pd.read_csv(OUT / 'resume_validation_discrepancies.csv', dtype=str)
display(disc.groupby('mismatched_fields').size().rename('profiles'))